# Implement Model Training

**How the data flow through the model during training:**
- During training, we use **Teacher Forcing** method.
  - **Input:** The Encoder receives the full Source sequence, and creates a context vector representation of it. # TODO correct?
  - **Shifted Target:** The Decoder receives the shifted Target sequence, e.g., if the goal is to predict "The brown rabbit", the Decoder receives `<SOS>, The, brown]`.
  - **Masking:** The padding mask ensures the model ignores the `<PAD>` tokens, and specifically the **Masked Multi-Head Attention** will ignore both the `<PAD>`, and the future tokens, e.g., (same example as shifted target) when the model is predicting "brown", it can't peek at the answer "rabbit".
  - **Decoder's output:** Produces a vector if size $d_{model} = 512$ for every token position.
  - **Generator:** *(The last two layers: linear→softmax)* take those $512$-dim vectors and projects them on the size of the vocabulary (tokenizer=37000).

In [22]:
# TODO the warmup in the markdown below is more complicated than it needs to be to be explained.


-  **Adam optimizer**: "We used the Adam optimizer [20] with $\beta_1 = 0.9$, $\beta_2 = 0.98$, and $\epsilon = 10^{−9}$"
   -  $$ \text{lrate} = d^{-0.5}_{\text{model}} * \text{min} (\text{stepNum}^{-0.5}, \quad \text{stepNum} * \text{warmupSteps}^{-1.5}) $$
      -  Example: $\quad d_{model} = 512,\quad \text{stepNum} = 100000,\quad \text{warmupSteps} = 4000$
         - $ \text{lrate} = 512^{-0.5} * \text{min}(100000 * 4000^{-1.5}) = 0.000139...$
           - The $\text{stepNum} = 100000$ is the **current step** (e.g., step 1, step 2, ...) it is a counter that ticks every time a single batch is processed.
           - Paper: "We trained the base models for a total of 100,000 steps or 12 hours." This means that they stopped training after step $100000$.
             - So $ 0.000139...$ is the learning rate at step $100000$
   -  "This corresponds to increasing the learning rate [$\text{lrate}$] linearly for the first $\text{warmupSteps}$ training steps, and decreasing it thereafter proportionally to the inverse square root of the step number. We used $\text{warmupSteps} = 4000$."
      -  **Warmup**: For the first $4000$ steps, the $\text{stepNum} * \text{warmupSteps}^{-1.5}$ term is smaller, this causes the learning rate to increase linearly, which "warms-up" the model, preventing the gradients from exploding early on when gradients are random.
      -  **Decay** $\text{min}( \text{stepNum}^{-0.5} > ...)$: After $4000$ steps, the $\text{stepNum}^{-0.5}$ is smaller inside the $\text{min}()$. This causes the learning rate to decrease following the inverse square root.  
         -  Paper: "This corresponds to increasing the learning rate linearly for the first $\text{warmupSteps}$ training steps, and decreasing it thereafter proportionally to the inverse square root of the step number. We used $\text{warmupSteps} = 4000$."
      -  The dim ($d^{-0.5}_{model}$): This scales the entire learning rate based on the model size.
  -  Note: that these $\quad d_{model} = 512,\quad \text{stepNum} = 100000,\quad \text{warmupSteps} = 4000$ is what the paper used to train on the full 4.5 million sentence from the **WMT14 dataset**, we will not use the entire dataset, the lrate will be updated differently depending on the size of the dataset.
  - **Dataset Size Adjustment:** 
    - **Example:** If we were to train on only $45088$ ($1$%) of the WMT14 en-de dataset with a batch_size $= 64$, than $45088/64 = 704$ **steps per epoch**.
       - The $= 704$ means the optimizer will update the model's weights $= 704$ times per epoch.
       - **Formulas:**
         - **1 Step (or iteration):** One forward pass + one backward pass on **one batch**.
         - **Steps Per Epoch** (i.e., $\text{warmupSteps}) =$ Total_sentence_pairs $\div$ batch_size
         - **Total Training Steps:** Steps per epoch $\times$ Total Epochs
         - So, in this example set $\text{warmupSteps} = 704$ so that the learning rate starts to decrease at about the end of the first epoch, which coincides with the end of the warmup and the model can start training. 
           - Result: 
             - Batch 1 & $\text{stepNum}$ = 1, learning rate is small.
             - Batch 704 & $\text{stepNum}$ = 704, learning rate is at its **peak** (end of epoch 1)
             - Batch 705 & $\text{stepNum}$ = 705, learning rate begins to **decay** (start of epoch 2)
       - This example is shown in `lrate_growth_example()` below


**Notes:**
- The paper used a fixed number of **tokens per batch**, not a fixed number of sentence pairs → "Each training batch contained a set of sentence pairs containing approximately $25000$ source tokens and $25000$ target tokens."

- **Validation**: check [beam_search.ipynb](./beam_search.ipynb)

- **Loss function**
  - **Regularization**:
    - **Label Smoothing**: "During training, we employed label smoothing of value $\in_{ls} = 0.1$. This hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score" to the loss function during training loop.
      - Its added to the loss

In [23]:
# <!-- #TODO make sure formulas are displayed correct on the repo -->

In [24]:
import torch.nn as nn
import torch
from torch.optim.lr_scheduler import LambdaLR
from datetime import datetime


try:  # works when ran via main.py (package mode)
    from .utils import make_target_mask
except ImportError:
    # Works when running from inside Jupyter Notebook
    from utils import make_target_mask

In [25]:
from typing import TYPE_CHECKING

if TYPE_CHECKING:  # for type checks example cfg: English_german_config below
    from .Transformer import Transformer
    from ..configs import English_german_config
    from tokenizers import Tokenizer

## Batch

In [26]:
class Batch:
    def __init__(self, src, tgt=None, pad_token: int = 0):
        """
        Handles token masking logic.

        Args:
            src = A batch of tokenized Source Sequence sentences. Example: if batch_size=64, than 
                `src` holds 64 English sentences and `tgt` holds the corresponding 64 German sentences.
            tgt = A batch of tokenized Target Sequence.
            pad_token: The integer representation for the '<PAD>' token
        """

        # TODO understand this?
        """Object to hold a batch of data with mask during training."""
        self.src = src
        self.src_padding_mask = (src != pad_token).unsqueeze(-2).unsqueeze(-2)

        self.tgt = tgt

        if tgt is not None:
            # Previous context: Take all tokens excerpt for the last one, this includes the <SOS> at the start.
            self.tgt = tgt[:, :-1]

            # 🌟 OUTPUTS (SHIFTED RIGHT): We take all tokens except for the first one (<SOS>), this start with the first actual word and ends with <EOS>
            self.tgt_y = tgt[:, 1:] # this is what the model is trying to predict at each time step

            self.tgt_no_peek_mask = make_target_mask(self.tgt, pad_token)

            # non_tokens: Used to divide the total loss by the number of non-padding tokens.
            self.non_tokens = (self.tgt_y != pad_token).data.sum()

## Learning Rate Schedule

In [27]:
def get_std_opt(model: Transformer, d_model=512, warmup_steps=4_000):
    """
    Get the scheduler and optimizer.
    """
    # TODO understand this?
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1,  # lr=1 so the scheduler controls the absolute value.
        betas=(0.9, 0.98),
        eps=1e-9,
    )

    lr_scheduler = LambdaLR(
        optimizer=optimizer,
        lr_lambda=lambda step: lrate(
            step, d_model, factor=1, warmup_steps=warmup_steps
        ),
    )
    return optimizer, lr_scheduler


def lrate(step_num, d_model, warmup_steps, factor=1.0):
    """The lrate formula as shown in the top formula.
    
    Args:
        step_num: The current step.
        d_model: Size of the model.
        factor: Default to 1.0. If the model is learning to slowly, increase to (factor=2.0 or higher), if the loss is exploding becoming NaN, lower factor (e.g., 0.5).
        warmup_steps: How many steps to "warmup" the model.
    """
    if step_num == 0:
        step_num = 1
    return factor * (
        d_model ** (-0.5) * min(step_num ** (-0.5), step_num * warmup_steps ** (-1.5))
    )

In [28]:
def lrate_growth_example():
    batch_size = 64
    total_sentences = 45088
    d_model = 512
    # Calculate how many steps (batches) make up one epoch
    steps_per_epoch = total_sentences // batch_size
    warmup = steps_per_epoch

    print(f"Lrate example on 1% of WMT 14 dataset")
    print(f"Batch size: {batch_size}")
    print(f"Total sentences in dataset: {total_sentences:,}")
    print(f"Steps per epoch: {steps_per_epoch:,}")
    print(f"Warmup ends at step: {warmup} (End of epoch 1)")
    print(
        f"\n\n{'Step':>8} | {'Approx Epoch':>12} | {'Sentences Seen':>16} | {'Learning-Rate':>12} | {'Phase'}"
    )

    milestones = [
        1,
        350,
        704,
        705,
        1500,
        3000,
        5000,
    ]  # Where the learning rate is changed.

    for step in milestones:
        lr = lrate(step_num=step, d_model=d_model, factor=1.0, warmup_steps=warmup)

        # Calculate progress
        curr_epoch = step / steps_per_epoch
        sentences_seen = step * batch_size

        if step <= warmup:
            phase = "Warmup (Linear up ↑)"
        else:
            phase = "Decay (Inverse Square Root ↓)"

        print(
            f"{step:8d} | {curr_epoch:12.2f} | {sentences_seen:16,d} | {lr:12.8f} | {phase}"
        )


# lrate_growth_example()

## Losss

In [ ]:
if TYPE_CHECKING:
    from .generator import Generator


class SimpleLossCompute:
    def __init__(self, generator: Generator, criterion, opt=None):
        """
        Compute a simple loss function

        Args:
            generator: The last Linear → softmax layers
            opt: Adam optimizer
        """

        self.generator = generator
        self.criterion = criterion
        self.opt = opt

    def __call__(self, x, y, norm):
        loss = (
            self.criterion(x.contiguous().view(-1, x.size(-1)), y.contiguous().view(-1))
            / norm
        )

        loss.backward()
        if self.opt is not None:
            self.opt.step()
            self.opt.zero_grad()  # Clear gradients before the next batch.
        return loss.item() * norm # TODO was return loss.data * norm is it fixed now?

## Train Model

In [ ]:
import os


class TrainModel(nn.Module):
    def __init__(self, cfg: English_german_config, model: Transformer, device):

        super().__init__()
        self.cfg = cfg
        self.model = model
        self.device = device

        self.criterion = nn.CrossEntropyLoss(
            label_smoothing=0.1,  # Used Torch's built-in Label Smoothing.
            ignore_index=cfg.special_tokens["pad_token"],
            reduction="sum",
        )

        self.optimizer, self.scheduler = get_std_opt(
            model=model,
            d_model=cfg.d_model,
            warmup_steps=cfg.warmup_steps,
        )
        self.compute_loss = SimpleLossCompute(
            model.generator, self.criterion, self.optimizer
        )

        # Track the steps. Once it reaches `step_num_limit` we end training
        self.step_counter = 0

    def save_checkpoint(self, epoch, avg_loss):
        checkpoint_name = (
            f"transformer_epoch_{epoch+1}_{self.cfg.perc_to_download}_percent_ds.pt"
        )
        checkpoint_path = os.path.join(
            self.cfg.MODEL_DIR, "checkpoints", checkpoint_name
        )
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": self.model.state_dict(),
                "optimizer_state_dict": self.optimizer.state_dict(),
                "scheduler_state_dict": self.scheduler.state_dict(),
                "step_counter": self.step_counter,
                "loss": avg_loss,
            },
            checkpoint_path,
        )
        print(f"Saved Checkpoint to -> {checkpoint_path}")

    def train(self, train_dataloader, start_epoch):
        """
        Train a model
        Args:
            start_epoch: Will depend on if we are training from a checkpoint or training a new model.
        """
        num_epochs = self.cfg.num_epochs
        print("\n" + "#" * 64)
        print(f"\nTraining Model")
        print(f"Num epochs: {num_epochs} | device: {self.device}")
        print("\n" + "#" * 64)

        for epoch in range(start_epoch, num_epochs):
            avg_loss = self.run_epoch(train_dataloader)
            print(
                f"Epoch [{epoch+1}/{num_epochs}] completed. Average Loss: {avg_loss:.4f}"
            )

            if self.step_counter >= self.cfg.step_num_limit:
                print(
                    f"Reached step limit: {self.cfg.step_num_limit}. Final Checkpoint..."
                )
                self.save_checkpoint(epoch=epoch, avg_loss=avg_loss)
                return

            self.save_checkpoint(epoch=epoch, avg_loss=avg_loss)

        print("Training complete!\n\n")

    def run_epoch(self, dataloader):
        """Run a single epoch"""
        total_tokens = 0
        total_loss = 0
        device = self.device
        self.model.train()

        for i, batch in enumerate(dataloader):

            if self.step_counter >= self.cfg.step_num_limit:
                return total_loss / (total_tokens if total_tokens > 0 else 1)

            # Move batch to device
            src = batch.src.to(device)
            tgt = batch.tgt.to(device)
            tgt_y = batch.tgt_y.to(device)
            src_padding_mask = batch.src_padding_mask.to(device)
            tgt_no_peek_mask = batch.tgt_no_peek_mask.to(device)

            # Forward Pass (The model returns decoder output before the generator (last linear + softmax layers))
            output = self.model(src, tgt, src_padding_mask, tgt_no_peek_mask)

            # Loss compute, performs backward prop
            loss = self.compute_loss(output, tgt_y, batch.non_tokens)

            total_loss += loss.item()
            total_tokens += batch.non_tokens

            # Update Learning Rate per step
            self.scheduler.step()
            self.step_counter += 1

            if i % 50 == 0:
                print(
                    f"{datetime.now().strftime('%m-%d %H:%M:%S')} | Step: {i} | Loss: {loss/batch.non_tokens:.4f} | Tokens: {total_tokens}"
                )

        return total_loss / total_tokens